# Solver scaling study

Compare discounted VI and MPI as the state count grows within one Garnet family, with fixed action count, transition density, discount and final precision. This small example uses the current `ScalingBenchmark` API; it does not implement the planned library extensions in the [roadmap](../docs/roadmap.md#scalingbenchmark-extensions) or [requirements](../docs/requirements.md#scaling-studies).

Use the `benchmark` Conda kernel with mdpforge installed from this checkout, pandas and IPython available. Run cells in order. The notebook is saved without measurements; running it starts actual solver trials and writes CSV/manifest pairs under `artifacts/tmp/scaling/`. Increase sizes only after inspecting the small study.

In [ ]:
import os

THREADS = 1
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = str(THREADS)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from threadpoolctl import threadpool_limits

from mdpforge import BenchmarkResult, ScalingBenchmark
from mdpforge.models.discounted.garnet import Model as Garnet

## Fixed protocol

Model seeds generate distinct transition instances; trial seeds control solver randomness and are paired by the benchmark. Keep the ordered size list and instance count fixed between conditions because the current API allocates model seeds from that order. Garnet's built-in reward generator uses its own fixed seed, so rewards are identical across transition instances at a given size. Its transition density is 0.85: transition storage grows approximately quadratically in the state count, rather than keeping a fixed successor count.

There are no warm-up trials. The first trial can include cold initialization; later trials can reuse process or system state. Interpret the recorded repeat order accordingly. Runtime includes construction, backend conversions and solver execution. Certification runs afterward and can affect subsequent allocation and cache behavior. All conditions in this notebook use the same validation mode.

In [ ]:
SIZES = [100, 200, 400]
INSTANCES = 3
ACTIONS = 4
MODEL_SEED = 100
TRIAL_SEED = 0
DISCOUNT = 0.9
PRECISION = 1e-3
REPEATS = 3
TIMEOUT = 60
VALIDATE = True
REFERENCE = "mdpforge_vi"
MPI_OPTIONS = {"max_evaluation_iterations": 10, "evaluation_precision": 1e-3}

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "src/mdpforge").is_dir()
)
OUTPUT_DIR = (
    REPO_ROOT
    / "artifacts/tmp/scaling"
    / datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_%f")
)

## Build reproducible instances

The factory builds the catalogue model using its existing recipe. It seeds the legacy NumPy generator used by Garnet's transition construction, then restores its previous state even on failure. The model name includes the generation seed. No model or optimal-value cache is populated. This does not change the catalogue implementation or its fixed reward seed.

In [ ]:
def make_garnet(size, model_seed):
    """Build a Garnet transition instance while preserving global RNG state."""
    model = Garnet(state_dim=size, action_dim=ACTIONS)
    model.name = f"{model.name}_seed_{model_seed}"
    previous_state = np.random.get_state()
    try:
        np.random.seed(model_seed)
        model.create_model(cache="off")
    finally:
        np.random.set_state(previous_state)
    return model


bench = ScalingBenchmark(criterion="discounted")
bench.add_family(
    make_garnet, SIZES, instances=INSTANCES, seed=MODEL_SEED, family="Garnet"
)
bench.add_solver(REFERENCE)
bench.add_solver("mdpforge_mpi", **MPI_OPTIONS)

## Run and save one condition

The current engine holds all family instances in memory. Model construction above is outside solver timing. A future progressive mode must be an explicit protocol choice. The timeout uses worker processes for the importable catalogue model and solvers; certification stays in the parent process outside that timeout.

In [ ]:
def run_condition(discount):
    """Run one discount condition and save its CSV and manifest."""
    with threadpool_limits(limits=THREADS):
        result = bench.run(
            discount=discount,
            precision=PRECISION,
            repeats=REPEATS,
            seed=TRIAL_SEED,
            timeout=TIMEOUT,
            validate=VALIDATE,
            verbose=True,
        )
    result.experiment["study"]["notebook_protocol"] = {
        "thread_limit": THREADS,
        "transition_seed": "legacy NumPy seeded per instance; previous state restored",
        "reward_seed": 0,
        "transition_density": 0.85,
        "warmup_trials": 0,
        "execution_mode": "all_instances_built_before_trials",
    }
    path = result.export_csv(OUTPUT_DIR / f"{result.experiment['experiment_id']}.csv")
    print(f"discount={discount}: {path}")
    return result, path


result, csv_path = run_condition(DISCOUNT)

## Execution coverage and instance dispersion

Each instance contributes its median trial runtime only when every expected repeat succeeds. The summary then aggregates those instance medians. The quartiles describe differences between transition instances, not repeat noise or confidence intervals. Current summary and speedup eligibility depend on execution success, not certification; read the separate quality table below before drawing accuracy claims.

In [ ]:
summary = result.scaling_summary(reference=REFERENCE)
display(summary)

fig, ax = result.plot_scaling(reference=REFERENCE, show=False)
for solver, group in summary.groupby("solver", sort=False):
    group = group.sort_values("state_dim")
    label = result.experiment["solvers"][solver]["metadata"]["display_name"]
    line = next(line for line in ax.lines if line.get_label() == label)
    ax.fill_between(
        group["state_dim"].to_numpy(),
        group["q25_runtime"].to_numpy(),
        group["q75_runtime"].to_numpy(),
        color=line.get_color(),
        alpha=0.15,
    )
plt.show()

_ = result.plot_scaling(metric="speedup", reference=REFERENCE)
_ = result.plot_scaling(x="nnz", reference=REFERENCE)

## Certification and raw trials

Execution success alone does not certify accuracy. The following counts are trial counts, grouped by actual size and solver, with execution and validation kept separate. The raw table also exposes repeat-to-repeat runtime variation. A library summary of fully certified instances and a certified-only paired comparison remain planned extensions; no automatic quality filtering is applied here.

The `nnz` plot above uses the median transition nonzeros within each state-size group. It is not a controlled sweep of successors or action count. A certified error bound is a guarantee from the recorded certificate, not a measured error against an exact solution.

In [ ]:
trials = result.to_dataframe()
display(
    trials.groupby(["state_dim", "solver", "status", "validation_status"])
    .size()
    .rename("n_trials")
    .reset_index()
)
display(
    trials[
        [
            "model",
            "state_dim",
            "solver",
            "repeat",
            "seed",
            "status",
            "runtime",
            "validation_status",
            "validation_runtime",
            "error_bound",
        ]
    ].sort_values(["state_dim", "model", "solver", "repeat"])
)
_ = result.plot_status()

## Reload saved evidence

Pass a specific CSV path to inspect a previous campaign. Loading and plotting saved results does not execute solvers. The companion `.experiment.json` must stay with its CSV for scaling analysis.

In [ ]:
saved = BenchmarkResult.from_csv(csv_path)
display(saved.scaling_summary(reference=REFERENCE))
print("Manifest:", csv_path.with_suffix(".experiment.json"))

## Optional discount comparison

Leave the tuple empty for the initial study. For example, `(0.95, 0.99)` requests two additional campaigns. These reuse the same already-built model matrices, solver options, repeat plan, thread limit and validation mode; only discount changes, while the absolute final precision stays fixed. Each condition has its own CSV and manifest. Curves below describe execution runtimes; compare their separate quality and coverage tables before interpreting a difference.

This example guarantees shared instances by reusing `bench`; it does not provide the planned compatibility checks for arbitrary saved runs. To study final precision instead, design a separate controlled campaign and record that intended change. Generalized action/successor sweeps and progressive execution are also future work. No memory profile or asymptotic complexity estimate is inferred from these timings.

In [ ]:
ADDITIONAL_DISCOUNTS = ()
conditions = {DISCOUNT: (result, csv_path)}
for discount in ADDITIONAL_DISCOUNTS:
    if discount not in conditions:
        conditions[discount] = run_condition(discount)

if len(conditions) > 1:
    fig, ax = plt.subplots(figsize=(8, 5))
    for discount, (condition, path) in conditions.items():
        stats = condition.scaling_summary(reference=REFERENCE)
        display(stats)
        display(
            condition.to_dataframe()
            .groupby(["state_dim", "solver", "status", "validation_status"])
            .size()
            .rename("n_trials")
            .reset_index()
        )
        for solver, group in stats.groupby("solver", sort=False):
            group = group.sort_values("state_dim")
            ax.plot(
                group["state_dim"],
                group["median_runtime"],
                marker="o",
                label=f"{solver}, discount={discount}",
            )
    ax.set(
        xscale="log",
        yscale="log",
        xlabel="States",
        ylabel="Median runtime per completed instance (s)",
    )
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()